<a href="https://colab.research.google.com/github/Sajjubuoy/CODEBODH/blob/main/FineTuned_CODEBODH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CODEBODH AI TUTOR - COMPLETE SYSTEM
# ============================================================

# -----------------------------
# 1. INSTALL LIBRARIES
# -----------------------------

!pip install -q unsloth transformers accelerate peft bitsandbytes


# -----------------------------
# 2. IMPORT LIBRARIES
# -----------------------------

import unsloth
from unsloth import FastLanguageModel

import torch
import os
import sys
import ast
import traceback


# -----------------------------
# 3. CREATE PROJECT STRUCTURE
# -----------------------------

os.makedirs("codebodh/analysis", exist_ok=True)
os.makedirs("codebodh/mapping", exist_ok=True)
os.makedirs("codebodh/engine", exist_ok=True)
os.makedirs("codebodh/tutor", exist_ok=True)


# ============================================================
# 4. CODE ANALYZER
# ============================================================

with open("codebodh/analysis/analyzer.py","w") as f:
    f.write("""

import ast
import traceback

def analyze_code(code: str):

    try:
        ast.parse(code)

    except SyntaxError as e:

        return {
            "syntax_valid": False,
            "runtime_error": None,
            "error": {
                "type": "SyntaxError",
                "message": str(e),
                "line": e.lineno
            }
        }

    try:

        exec_globals = {}
        exec(code, exec_globals)

        return {
            "syntax_valid": True,
            "runtime_error": None,
            "error": None
        }

    except Exception as e:

        return {
            "syntax_valid": True,
            "runtime_error": True,
            "error": {
                "type": type(e).__name__,
                "message": str(e),
                "traceback": traceback.format_exc()
            }
        }

""")

    # ============================================================
# 4.5 SEMANTIC CODE ANALYZER (AST LOGIC DETECTION)
# ============================================================

with open("codebodh/analysis/semantic_analyzer.py","w") as f:
    f.write("""

import ast

def analyze_semantics(code):

    issues = []

    try:
        tree = ast.parse(code)
    except:
        return issues

    assigned_vars = set()
    used_vars = set()

    for node in ast.walk(tree):

        # Detect assigned variables
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    assigned_vars.add(target.id)

        # Detect variable usage
        if isinstance(node, ast.Name):
            used_vars.add(node.id)

        # Detect inefficient loops
        if isinstance(node, ast.For):
            if isinstance(node.iter, ast.Call):
                if hasattr(node.iter.func, "id") and node.iter.func.id == "range":
                    if len(node.iter.args) == 1:
                        issues.append({
                            "type": "Inefficient Loop",
                            "message": "Loop may be inefficient. Consider iterating directly over collection."
                        })

        # Detect recursion
        if isinstance(node, ast.FunctionDef):
            function_name = node.name

            for child in ast.walk(node):
                if isinstance(child, ast.Call):
                    if isinstance(child.func, ast.Name):
                        if child.func.id == function_name:
                            issues.append({
                                "type": "Recursion",
                                "message": "Recursive function detected. Ensure base condition exists."
                            })

    # Detect unused variables
    unused_vars = assigned_vars - used_vars

    for var in unused_vars:
        issues.append({
            "type": "Unused Variable",
            "message": f"Variable '{var}' assigned but never used."
        })

    return issues

""")


# ============================================================
# 5. ERROR → CONCEPT MAPPING
# ============================================================

# ============================================================
# 5. ERROR → CONCEPT MAPPING
# ============================================================

with open("codebodh/mapping/concept_mapper.py","w") as f:
    f.write("""

ERROR_CONCEPT_MAP = {

    "NameError": {
        "concept": "Variables",
        "subconcept": "Undefined Variable"
    },

    "UnboundLocalError": {
        "concept": "Variables",
        "subconcept": "Local Variable Referenced Before Assignment"
    },

    "TypeError": {
        "concept": "Data Types",
        "subconcept": "Type Mismatch"
    },

    "ValueError": {
        "concept": "Data Types",
        "subconcept": "Invalid Value for Data Type"
    },

    "ZeroDivisionError": {
        "concept": "Arithmetic",
        "subconcept": "Division by Zero"
    },

    "OverflowError": {
        "concept": "Arithmetic",
        "subconcept": "Number Overflow"
    },

    "FloatingPointError": {
        "concept": "Arithmetic",
        "subconcept": "Floating Point Calculation Error"
    },

    "IndexError": {
        "concept": "Lists",
        "subconcept": "Index Out of Range"
    },

    "KeyError": {
        "concept": "Dictionaries",
        "subconcept": "Key Not Found"
    },

    "AttributeError": {
        "concept": "Objects",
        "subconcept": "Attribute Not Found"
    },

    "ImportError": {
        "concept": "Modules",
        "subconcept": "Module Import Failed"
    },

    "ModuleNotFoundError": {
        "concept": "Modules",
        "subconcept": "Module Not Installed or Not Found"
    },

    "SyntaxError": {
        "concept": "Python Syntax",
        "subconcept": "Invalid Syntax"
    },

    "IndentationError": {
        "concept": "Python Syntax",
        "subconcept": "Incorrect Indentation"
    },

    "TabError": {
        "concept": "Python Syntax",
        "subconcept": "Mixed Tabs and Spaces"
    },

    "RuntimeError": {
        "concept": "Runtime Execution",
        "subconcept": "Generic Runtime Error"
    },

    "RecursionError": {
        "concept": "Recursion",
        "subconcept": "Maximum Recursion Depth Exceeded"
    },

    "MemoryError": {
        "concept": "Memory Management",
        "subconcept": "Out of Memory"
    },

    "AssertionError": {
        "concept": "Assertions",
        "subconcept": "Assertion Condition Failed"
    },

    "StopIteration": {
        "concept": "Iterators",
        "subconcept": "Iterator Exhausted"
    },

    "StopAsyncIteration": {
        "concept": "Async Iterators",
        "subconcept": "Async Iterator Exhausted"
    },

    "GeneratorExit": {
        "concept": "Generators",
        "subconcept": "Generator Exit Triggered"
    },

    "FileNotFoundError": {
        "concept": "File Handling",
        "subconcept": "File Not Found"
    },

    "FileExistsError": {
        "concept": "File Handling",
        "subconcept": "File Already Exists"
    },

    "PermissionError": {
        "concept": "File Handling",
        "subconcept": "Permission Denied"
    },

    "IsADirectoryError": {
        "concept": "File Handling",
        "subconcept": "Expected File But Found Directory"
    },

    "NotADirectoryError": {
        "concept": "File Handling",
        "subconcept": "Expected Directory But Found File"
    },

    "BlockingIOError": {
        "concept": "Input Output",
        "subconcept": "Blocking IO Operation"
    },

    "BrokenPipeError": {
        "concept": "Input Output",
        "subconcept": "Broken Pipe Communication"
    },

    "BufferError": {
        "concept": "Memory Buffers",
        "subconcept": "Buffer Operation Failed"
    },

    "ChildProcessError": {
        "concept": "Process Management",
        "subconcept": "Child Process Error"
    },

    "ConnectionError": {
        "concept": "Networking",
        "subconcept": "General Connection Error"
    },

    "ConnectionAbortedError": {
        "concept": "Networking",
        "subconcept": "Connection Aborted"
    },

    "ConnectionRefusedError": {
        "concept": "Networking",
        "subconcept": "Connection Refused"
    },

    "ConnectionResetError": {
        "concept": "Networking",
        "subconcept": "Connection Reset"
    },

    "TimeoutError": {
        "concept": "Networking",
        "subconcept": "Operation Timed Out"
    },

    "InterruptedError": {
        "concept": "System Calls",
        "subconcept": "Interrupted System Call"
    },

    "EOFError": {
        "concept": "Input Handling",
        "subconcept": "Unexpected End of File"
    },

    "KeyboardInterrupt": {
        "concept": "Program Control",
        "subconcept": "Manual Program Interruption"
    },

    "NotImplementedError": {
        "concept": "Object Oriented Programming",
        "subconcept": "Method Not Implemented"
    },

    "OSError": {
        "concept": "Operating System",
        "subconcept": "OS Level Error"
    },

    "ReferenceError": {
        "concept": "Memory Management",
        "subconcept": "Invalid Weak Reference"
    },

    "SystemError": {
        "concept": "Python Interpreter",
        "subconcept": "Internal Interpreter Error"
    },

    "SystemExit": {
        "concept": "Program Termination",
        "subconcept": "Program Exit Triggered"
    },

    "UnicodeError": {
        "concept": "Strings",
        "subconcept": "Unicode Encoding Issue"
    },

    "UnicodeEncodeError": {
        "concept": "Strings",
        "subconcept": "Unicode Encoding Failed"
    },

    "UnicodeDecodeError": {
        "concept": "Strings",
        "subconcept": "Unicode Decoding Failed"
    },

    "UnicodeTranslateError": {
        "concept": "Strings",
        "subconcept": "Unicode Translation Failed"
    }

}


def map_error_to_concept(error_info):

    if not error_info:
        return None

    error_type = error_info.get("type")

    if error_type in ERROR_CONCEPT_MAP:

        mapped = ERROR_CONCEPT_MAP[error_type]

        return {
            "concept": mapped["concept"],
            "subconcept": mapped["subconcept"],
            "confidence": 0.9
        }

    return {
        "concept": "Unknown",
        "subconcept": "Unmapped Error",
        "confidence": 0.3
    }

""")


# ============================================================
# 6. DIAGNOSTIC ENGINE
# ============================================================

with open("codebodh/engine/diagnostic_engine.py","w") as f:
    f.write("""

from analysis.analyzer import analyze_code
from analysis.semantic_analyzer import analyze_semantics
from mapping.concept_mapper import map_error_to_concept

def run_diagnostic(code):

    analysis_result = analyze_code(code)

    semantic_issues = analyze_semantics(code)

    concept_result = None

    if analysis_result["error"]:
        concept_result = map_error_to_concept(analysis_result["error"])

    return {
        "analysis": analysis_result,
        "concept": concept_result,
        "semantic_issues": semantic_issues
    }

""")


# ============================================================
# 7. PROMPT BUILDER
# ============================================================

with open("codebodh/tutor/prompt_builder.py","w") as f:
    f.write("""

def build_pedagogical_prompt(student_code, diagnostic_output):

    analysis = diagnostic_output.get("analysis")
    concept = diagnostic_output.get("concept")
    semantic = diagnostic_output.get("semantic_issues")

    error_type = None
    error_message = None

    if analysis["error"]:
        error_type = analysis["error"]["type"]
        error_message = analysis["error"]["message"]

    prompt = f'''

You are a disciplined Python programming tutor.

A student wrote the following code:

{student_code}

'''

    if error_type:
        prompt += f'''
The program produced this error:

Error Type: {error_type}
Error Message: {error_message}

'''

    if concept:
        prompt += f'''
Concept involved:
Main Concept: {concept['concept']}
Subconcept: {concept['subconcept']}
'''

    if semantic:
        prompt += f'''

Potential Logical Issues Detected:

{semantic}

'''

    prompt += '''

Teach the student using EXACTLY this format:

Concept Diagnosis
Explain the concept causing the mistake.

Probing Questions
Ask 2 guiding questions.

Hint
Give a small hint but not the answer.

Reflection
Encourage the student to think about the mistake.

'''

    return prompt

""")


# ============================================================
# 8. LLM RESPONSE GENERATOR
# ============================================================

with open("codebodh/tutor/llm_generator.py","w") as f:
    f.write("""

import torch

def generate_llm_response(prompt, model, tokenizer):

    formatted_prompt = f"<s>[INST] {prompt} [/INST]"

    inputs = tokenizer(formatted_prompt, return_tensors="pt")

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k,v in inputs.items()}

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.3,
            top_p=0.9,
            do_sample=True
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return decoded.split("[/INST]")[-1].strip()

""")


# ============================================================
# 9. IMPORT PIPELINE
# ============================================================
# ============================================================
# 9. IMPORT PIPELINE
# ============================================================

# Add project path
sys.path.append("/content/codebodh")

# Import modules
import importlib

import engine.diagnostic_engine
import tutor.prompt_builder
import tutor.llm_generator

# Reload modules to ensure latest versions are used
importlib.reload(engine.diagnostic_engine)
importlib.reload(tutor.prompt_builder)
importlib.reload(tutor.llm_generator)

# Import functions after reload
from engine.diagnostic_engine import run_diagnostic
from tutor.prompt_builder import build_pedagogical_prompt
from tutor.llm_generator import generate_llm_response

print("✅ CodeBodh pipeline loaded successfully")

# ============================================================
# CLEAR GPU MEMORY BEFORE MODEL LOAD
# ============================================================

import torch
import gc

gc.collect()
torch.cuda.empty_cache()

print("✅ GPU memory cleared")


# ============================================================
# 10. LOAD MISTRAL MODEL
# ============================================================

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.1-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True
)

model.eval()

print("✅ Base model loaded")


# ============================================================
# 11. LOAD YOUR CODEBODH ADAPTER
# ============================================================

# -----------------------------
# STEP 1 — Check zip file exists
# -----------------------------
!ls /content

# -----------------------------
# STEP 2 — Unzip model
# -----------------------------
!unzip -o /content/codebodh_model-20260305T123243Z-1-001.zip -d /content/codebodh_adapter

# -----------------------------
# STEP 3 — Check extracted files
# -----------------------------
!ls /content/codebodh_adapter
!ls /content/codebodh_adapter/codebodh_model

# -----------------------------
# STEP 4 — Load adapter
# -----------------------------
from peft import PeftModel

model = PeftModel.from_pretrained(
    model,
    "/content/codebodh_adapter/codebodh_model"
)

print("✅ CodeBodh adapter loaded successfully")

# -----------------------------
# STEP 5 — Move to GPU
# -----------------------------
model = model.to("cuda")
model.eval()

# -----------------------------
# STEP 6 — Test model
# -----------------------------
prompt = "Explain the Python error: NameError: name 'num3' is not defined."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


# ============================================================
# 12. TEST THE SYSTEM
# ============================================================

student_code = """

num1 = 10
num2 = 20

print(num1 + num3)

"""

print("STUDENT CODE:")
print(student_code)


# Run diagnostic
diagnostic_output = run_diagnostic(student_code)

print("\nDIAGNOSTIC OUTPUT:")
print(diagnostic_output)


# Build prompt
prompt = build_pedagogical_prompt(student_code, diagnostic_output)

print("\nPROMPT SENT TO MODEL:")
print(prompt)


# Generate AI tutor response
response = generate_llm_response(prompt, model, tokenizer)

print("\nCODEBODH RESPONSE:")
print(response)

✅ CodeBodh pipeline loaded successfully
✅ GPU memory cleared
==((====))==  Unsloth 2026.3.3: Fast Mistral patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Base model loaded
codebodh				   sample_data
codebodh_model-20260305T123243Z-1-001.zip  unsloth_compiled_cache
huggingface_tokenizers_cache
Archive:  /content/codebodh_model-20260305T123243Z-1-001.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/codebodh_model-20260305T123243Z-1-001.zip or
        /content/codebodh_model-20260305T123243Z-1-001.zip.zip, and cannot find /content/codebodh_model-20260305T123243Z-1-001.zip.ZIP, period.
ls: cannot access '/content/codebodh_adapter': No such file or directory
ls: cannot access '/content/codebodh_adapter/codebodh_model': No such file or directory


ValueError: Can't find 'adapter_config.json' at '/content/codebodh_adapter/codebodh_model'

In [ ]:
# ============================================================
# CODEBODH SIMPLE STUDENT INTERFACE
# ============================================================

import gradio as gr

def codebodh_interface(student_code):

    if student_code.strip() == "":
        return "Please enter Python code."

    # Run diagnostic
    diagnostic_output = run_diagnostic(student_code)

    # Build prompt
    prompt = build_pedagogical_prompt(student_code, diagnostic_output)

    # Generate tutor response
    response = generate_llm_response(prompt, model, tokenizer)

    return response


interface = gr.Interface(
    fn=codebodh_interface,

    inputs=gr.Textbox(
        lines=15,
        label="Paste Your Python Code Here"
    ),

    outputs=gr.Textbox(
        lines=20,
        label="CodeBodh AI Tutor"
    ),

    title="CodeBodh – AI Programming Tutor",

    description="Paste your Python code and get explanations, hints, and guidance."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6b5b93ecba8d495594.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
